In [ ]:
# Preprocessing comparison helpers
import cv2
import numpy as np
import torch
from pathlib import Path

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except Exception:
    YOLO_AVAILABLE = False

# default device for inference
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# global HSV saturation scale applied to all inputs (set >1.0 to boost)
HSV_S_SCALE = 1.75


def apply_clahe(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def apply_unsharp(frame, w_orig=1.2, w_blur=-0.2, sigma=1.0):
    blurred = cv2.GaussianBlur(frame, (0, 0), sigma)
    return cv2.addWeighted(frame, w_orig, blurred, w_blur, 0)


def apply_gamma(frame, gamma=1.2):
    inv = 1.0 / gamma
    table = (np.arange(256) / 255.0) ** inv * 255.0
    table = np.clip(table, 0, 255).astype('uint8')
    return cv2.LUT(frame, table)


def apply_hsv_saturation(frame, s_scale=1.5):
    """Increase HSV saturation by scale factor s_scale (>1 increases saturation).
    Keeps H and V channels unchanged; clips S to [0,255]."""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV).astype(np.float32)
    h, s, v = cv2.split(hsv)
    s = s * s_scale
    s = np.clip(s, 0, 255)
    hsv2 = cv2.merge((h, s, v)).astype(np.uint8)
    return cv2.cvtColor(hsv2, cv2.COLOR_HSV2BGR)


def apply_denoise(frame):
    return cv2.fastNlMeansDenoisingColored(frame, None, 10, 10, 7, 21)


def center_tile_upscale(frame, frac=0.5, scale=2):
    h, w = frame.shape[:2]
    tw, th = int(w * frac), int(h * frac)
    x1 = max(0, (w - tw) // 2)
    y1 = max(0, (h - th) // 2)
    tile = frame[y1:y1 + th, x1:x1 + tw]
    up = cv2.resize(tile, (tw * scale, th * scale), interpolation=cv2.INTER_CUBIC)
    out = frame.copy()
    down = cv2.resize(up, (tw, th), interpolation=cv2.INTER_AREA)
    out[y1:y1 + th, x1:x1 + tw] = down
    return out, up


def draw_label(img, text, pos=(10, 30), color=(0, 255, 255)):
    cv2.putText(img, text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)


def to_grayscale_display(img_color):
    """Return a 3-channel BGR image built from grayscale for display."""
    gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)


def run_compare(video_path, model_path=None, run_inference=False, max_frames=300, device=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Cannot open video: {video_path}")
        return

    model = None
    if run_inference and YOLO_AVAILABLE and model_path is not None:
        print(f"Loading model {model_path} ...")
        model = YOLO(model_path)
        # determine device to use for inference
        if device is None:
            device = DEVICE
        try:
            model.to(device)
        except Exception:
            pass

    paused = False
    frame_idx = 0

    while True:
        if not paused:
            ret, frame = cap.read()
            if not ret:
                print("End of video")
                break
            frame_idx += 1
            if frame_idx > max_frames:
                print("Reached max_frames, exiting")
                break

        # Original color frame (for display)
        orig = frame.copy()

        # Optionally apply global HSV saturation boost before all variants
        frame_proc = apply_hsv_saturation(frame, s_scale=HSV_S_SCALE) if HSV_S_SCALE and HSV_S_SCALE != 1.0 else frame

        # Create processed color variants (used for inference if enabled)
        clahe_c = apply_clahe(frame_proc)
        unsharp_c = apply_unsharp(frame_proc, w_orig=1.2, w_blur=-0.2, sigma=1.0)
        gamma_c = apply_gamma(frame_proc, gamma=1.2)
        hsv_up_c = apply_hsv_saturation(frame_proc, s_scale=1.5)
        denoise_c = apply_denoise(frame_proc)
        tile_merged_c, tile_up_c = center_tile_upscale(frame_proc, frac=0.5, scale=2)

        # For display, convert these variants to grayscale (3-channel) so you can compare clearly
        variants = [
            ("orig", orig, to_grayscale_display(orig)),
            ("clahe", clahe_c, to_grayscale_display(clahe_c)),
            ("unsharp", unsharp_c, to_grayscale_display(unsharp_c)),
            ("gamma", gamma_c, to_grayscale_display(gamma_c)),
            ("hsv_up", hsv_up_c, to_grayscale_display(hsv_up_c)),
            ("denoise", denoise_c, to_grayscale_display(denoise_c)),
            ("tile_merged", tile_merged_c, to_grayscale_display(tile_merged_c))
        ]

        det_counts = {}
        if run_inference and model is not None:
            # Run inference on the color variants
            for name, img_color, _ in variants:
                try:
                    results = model.predict(img_color, conf=0.15, iou=0.45, imgsz=720, device=device)
                    count = 0
                    for r in results:
                        if getattr(r, 'boxes', None) is not None:
                            count += len(r.boxes)
                    det_counts[name] = count
                except Exception:
                    det_counts[name] = 'err'
        else:
            for name, _, _ in variants:
                det_counts[name] = '-'

        # Build tiled grayscale display (2 rows x 3 cols)
        h, w = orig.shape[:2]
        tw, th = w // 3, h // 2
        canvas = np.zeros((th * 2, tw * 3, 3), dtype=np.uint8)
        for i, (name, _img_color, img_display) in enumerate(variants):
            r = cv2.resize(img_display, (tw, th))
            y = (i // 3) * th
            x = (i % 3) * tw
            canvas[y:y+th, x:x+tw] = r
            draw_label(canvas, f"{name} | det={det_counts.get(name)}", (x+8, y+25))

        # Display upscaled tile in grayscale too
        tile_vis = tile_up_c if 'tile_up_c' in locals() and tile_up_c is not None else np.zeros((100,100,3),dtype=np.uint8)
        tile_vis_gray = to_grayscale_display(tile_vis)
        tile_vis_gray = cv2.resize(tile_vis_gray, (min(640, tile_vis_gray.shape[1]), min(360, tile_vis_gray.shape[0])))

        cv2.imshow("compare (grayscale)", canvas)
        cv2.imshow("tile_up (grayscale)", tile_vis_gray)

        key = cv2.waitKey(0 if paused else 1) & 0xFF
        if key == ord('q') or key == 27:
            break
        elif key == ord(' '):
            paused = not paused
        elif key == ord('s'):
            out_dir = Path("./compare_outputs")
            out_dir.mkdir(exist_ok=True)
            fname = out_dir / f"frame_{frame_idx:06d}.jpg"
            cv2.imwrite(str(fname), canvas)
            print(f"Saved {fname}")
        elif key == ord('i'):
            run_inference = not run_inference
            print(f"run_inference = {run_inference}")

    cap.release()
    cv2.destroyAllWindows()


In [25]:
# Extract frames from video, run inference on each frame, save annotated images
from pathlib import Path

# Video file (change to your file)
video_path = Path('VIDEO.MOV')
model_path = Path('./weights/best.pt')
run_inference = True
max_frames = 1000  # max frames to process (set lower if you want fewer)

# output
out_dir = Path('./results/inference_images')
out_dir.mkdir(parents=True, exist_ok=True)
log_lines = []

print('YOLO available:', YOLO_AVAILABLE)
print('Using device:', DEVICE)
print('Video path:', video_path)
print('Global HSV_S_SCALE =', HSV_S_SCALE)

if not video_path.exists():
    print('Video not found:', video_path)
else:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print('Cannot open video:', video_path)
    else:
        # load model once
        model = None
        if run_inference and YOLO_AVAILABLE and model_path.exists():
            print(f'Loading model {model_path} ...')
            model = YOLO(str(model_path))
            try:
                model.to(DEVICE)
            except Exception:
                pass

        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1
            if frame_idx > max_frames:
                break

            # apply global HSV saturation before any processing
            frame_proc = apply_hsv_saturation(frame, s_scale=HSV_S_SCALE) if HSV_S_SCALE and HSV_S_SCALE != 1.0 else frame

            det_count = 0
            if run_inference and model is not None:
                try:
                    results = model.predict(frame, conf=0.15, iou=0.45, imgsz=720, device=DEVICE)
                except Exception as e:
                    print('Inference error at frame', frame_idx, e)
                    results = []

                try:
                    for r in results:
                        boxes = getattr(r, 'boxes', None)
                        if boxes is None:
                            continue
                        # xyxy, conf, cls arrays
                        xyxy = None
                        confs = None
                        cls_ids = None
                        if hasattr(boxes, 'xyxy'):
                            try:
                                xyxy = boxes.xyxy.cpu().numpy()
                            except Exception:
                                xyxy = boxes.xyxy
                        if hasattr(boxes, 'conf'):
                            try:
                                confs = boxes.conf.cpu().numpy()
                            except Exception:
                                confs = boxes.conf
                        if hasattr(boxes, 'cls'):
                            try:
                                cls_ids = boxes.cls.cpu().numpy()
                            except Exception:
                                cls_ids = boxes.cls

                        if xyxy is not None:
                            for i, bb in enumerate(xyxy):
                                x1, y1, x2, y2 = map(int, bb[:4])
                                conf = float(confs[i]) if confs is not None else 0.0
                                cid = int(cls_ids[i]) if cls_ids is not None else 0
                                label = f'{cid}:{conf:.2f}'
                                cv2.rectangle(frame_proc, (x1, y1), (x2, y2), (0, 255, 0), 2)
                                cv2.putText(frame_proc, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
                                det_count += 1
                        else:
                            for r in results:
                                if getattr(r, 'boxes', None) is not None:
                                    det_count += len(r.boxes)
                except Exception:
                    # fallback to counting
                    det_count = 0
                    for r in results:
                        if getattr(r, 'boxes', None) is not None:
                            det_count += len(r.boxes)

            # save annotated frame (saved image is the processed one)
            out_path = out_dir / f'frame_{frame_idx:06d}.jpg'
            cv2.imwrite(str(out_path), frame_proc)
            log_lines.append(f'{out_path.name}\t{det_count}')
            if frame_idx % 50 == 0:
                print(f'Processed frame {frame_idx} — saved {out_path} — {det_count} detections')

        cap.release()
        # write log
        with open(out_dir / 'detection_log.txt', 'w', encoding='utf-8') as f:
            f.write('\n'.join(log_lines))
        print('Done. Annotated frames saved to', out_dir)


YOLO available: True
Using device: cuda
Video path: VIDEO.MOV
Global HSV_S_SCALE = 2.0
Loading model weights\best.pt ...

WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
0: 736x416 (no detections), 9.7ms
Speed: 10.3ms preprocess, 9.7ms inference, 0.6ms postprocess per image at shape (1, 3, 736, 416)

WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
0: 736x416 (no detections), 12.7ms
Speed: 2.9ms preprocess, 12.7ms inference, 0.8ms postprocess per image at shape (1, 3, 736, 416)

WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
0: 736x416 (no detections), 9.6ms
Speed: 3.4ms preprocess, 9.6ms inference, 0.7ms postprocess per image at shape (1, 3, 736, 416)

WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
0: 736x416 (no detections), 17.8ms
Speed: 2.2ms preprocess, 17.8ms inference, 0.8ms postprocess per image at shape (1, 3, 736, 416)

WARNING imgsz=[720] must be multiple of max stride 32, up

In [26]:
# Multi-variant analysis on a single image to try to detect drone
from pathlib import Path

# Place your image in the repo (e.g. 'input.jpg') or change this path
image_path = Path('D:\\AI\\AI-6\\results\\inference_images\\frame_000974.jpg')
model_path = Path('./weights/best.pt')

out_root = Path('./results/analysis_single_image')
out_root.mkdir(parents=True, exist_ok=True)

print('Global HSV_S_SCALE =', HSV_S_SCALE)

if not image_path.exists():
    print('Image not found:', image_path)
    print('Please place the image at', image_path, "or change the path in this cell.")
else:
    img = cv2.imread(str(image_path))
    if img is None:
        print('Failed to read image:', image_path)
    else:
        # apply global HSV saturation boost before variants
        img_base = apply_hsv_saturation(img, s_scale=HSV_S_SCALE) if HSV_S_SCALE and HSV_S_SCALE != 1.0 else img

        # variants to try (name, function that returns color image)
        def var_orig(im):
            return im
        def var_clahe(im):
            return apply_clahe(im)
        def var_unsharp(im):
            return apply_unsharp(im)
        def var_gamma(im):
            return apply_gamma(im)
        def var_denoise(im):
            return apply_denoise(im)
        def var_tile_up(im):
            _, up = center_tile_upscale(im, frac=0.5, scale=2)
            # place upscaled tile in center of a black canvas same size for inference
            h, w = im.shape[:2]
            canvas = np.zeros_like(im)
            uh, uw = up.shape[:2]
            sx = max(0, (w - uw)//2)
            sy = max(0, (h - uh)//2)
            canvas[sy:sy+uh, sx:sx+uw] = up
            return canvas
        def var_hsv_up(im, s_scale=1.8):
            return apply_hsv_saturation(im, s_scale=s_scale)

        variants = [
            ('orig', lambda im: var_orig(img_base)),
            ('clahe', lambda im: var_clahe(img_base)),
            ('unsharp', lambda im: var_unsharp(img_base)),
            ('gamma', lambda im: var_gamma(img_base)),
            ('denoise', lambda im: var_denoise(img_base)),
            ('tile_up', lambda im: var_tile_up(img_base)),
            ('hsv_up', lambda im: var_hsv_up(img_base, s_scale=1.8)),
        ]

        confs = [0.10, 0.15, 0.25]
        imgsz = 1280

        if not YOLO_AVAILABLE:
            print('Ultralytics YOLO not available in this environment; cannot run model.')
        elif not model_path.exists():
            print('Model not found:', model_path)
        else:
            print('Loading model', model_path, '-> device', DEVICE)
            model = YOLO(str(model_path))
            try:
                model.to(DEVICE)
            except Exception:
                pass

            summary = []
            for vname, vfunc in variants:
                img_var = vfunc(img_base)
                for conf in confs:
                    try:
                        results = model.predict(img_var, conf=conf, iou=0.45, imgsz=imgsz, device=DEVICE)
                    except Exception as e:
                        print('Inference error', vname, conf, e)
                        continue

                    det_count = 0
                    annotated = img_var.copy()
                    boxes_info = []
                    for r in results:
                        boxes = getattr(r, 'boxes', None)
                        if boxes is None:
                            continue
                        xyxy = None
                        confs_arr = None
                        cls_arr = None
                        if hasattr(boxes, 'xyxy'):
                            try:
                                xyxy = boxes.xyxy.cpu().numpy()
                            except Exception:
                                xyxy = boxes.xyxy
                        if hasattr(boxes, 'conf'):
                            try:
                                confs_arr = boxes.conf.cpu().numpy()
                            except Exception:
                                confs_arr = boxes.conf
                        if hasattr(boxes, 'cls'):
                            try:
                                cls_arr = boxes.cls.cpu().numpy()
                            except Exception:
                                cls_arr = boxes.cls

                        if xyxy is not None:
                            for i, bb in enumerate(xyxy):
                                x1, y1, x2, y2 = map(int, bb[:4])
                                c = float(confs_arr[i]) if confs_arr is not None else 0.0
                                cid = int(cls_arr[i]) if cls_arr is not None else 0
                                label = f'{cid}:{c:.2f}'
                                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0,255,0), 2)
                                cv2.putText(annotated, label, (x1, max(15, y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
                                boxes_info.append((x1,y1,x2,y2,cid,c))
                                det_count += 1
                        else:
                            for r in results:
                                if getattr(r, 'boxes', None) is not None:
                                    det_count += len(r.boxes)

                    out_sub = out_root / f'{vname}_conf{int(conf*100)}.jpg'
                    cv2.imwrite(str(out_sub), annotated)
                    summary.append((vname, conf, det_count, out_sub, boxes_info))
                    print(f'{vname} conf={conf:.2f} -> {det_count} detections -> saved {out_sub}')

            # Save a simple CSV summary
            with open(out_root / 'summary.txt', 'w', encoding='utf-8') as f:
                for vname, conf, det_count, out_sub, boxes_info in summary:
                    f.write(f'{vname}\t{conf}\t{det_count}\t{out_sub.name}\n')
            print('Finished analysis. Results in', out_root)


Global HSV_S_SCALE = 2.0
Loading model weights\best.pt -> device cuda

0: 1280x704 1 drone, 26.4ms
Speed: 12.8ms preprocess, 26.4ms inference, 3.5ms postprocess per image at shape (1, 3, 1280, 704)
orig conf=0.10 -> 1 detections -> saved results\analysis_single_image\orig_conf10.jpg

0: 1280x704 1 drone, 24.1ms
Speed: 12.6ms preprocess, 24.1ms inference, 5.2ms postprocess per image at shape (1, 3, 1280, 704)
orig conf=0.15 -> 1 detections -> saved results\analysis_single_image\orig_conf15.jpg

0: 1280x704 1 drone, 22.8ms
Speed: 14.9ms preprocess, 22.8ms inference, 3.2ms postprocess per image at shape (1, 3, 1280, 704)
orig conf=0.25 -> 1 detections -> saved results\analysis_single_image\orig_conf25.jpg

0: 1280x704 1 drone, 26.7ms
Speed: 12.4ms preprocess, 26.7ms inference, 2.8ms postprocess per image at shape (1, 3, 1280, 704)
clahe conf=0.10 -> 1 detections -> saved results\analysis_single_image\clahe_conf10.jpg

0: 1280x704 1 drone, 26.0ms
Speed: 14.2ms preprocess, 26.0ms inference,